<a id='toc0_'></a>
## Content Outline

### Intro
- [The Backbone Matrix](#toc1_)
- [Short standalone versions](#toc1_1_1_1_)
- [A display utility for the backbone matrix](#toc1_1_2_)

### Sequences
- [Riordan](#toc1_2_)
- [Catalan](#toc1_3_)
- [Motzkin](#toc1_4_)
- [Fibonacci](#toc1_5_)
- [Factorial](#toc1_6_)
- [Kolakoski](#toc1_7_)
- [Bell](#toc1_8_)
- [Fubini](#toc1_9_)
- [Central Binomial](#toc1_10_)
- [Subfactorial / Rencontres / Derangements](#toc1_11_)
- [Involutions (Young tableaux with n cells)](#toc1_12_)
- [Moebius](#toc1_13_)
- [Pell](#toc1_14_)
- [Jacobsthal](#toc1_15_)
- [Polya Trees](#toc1_16_)
- [Euler (alternating permutations, boustrophedon transform)](#toc1_17_)
- [Little Schroeder numbers](#toc1_18_)
- [Big Schroeder numbers](#toc1_19_)
- [Central Delannoy](#toc1_20_)
- [Sets of Lists](#toc1_21_)
- [Total Partitions / Ward Set](#toc1_22_)
- [Ward Cycle](#toc1_23_)
- [Bernoulli numbers](#toc1_24_)

# <a id='toc1_'></a>The Backbone Matrix

In [ ]:
from collections.abc import Generator
from fractions import Fraction as frac
from itertools import islice

type Seq = list[int]
type Matrix = list[Seq]
type SeqGenerator = Generator[int, None, None]
type MatGenerator = Generator[Seq, None, None]

def print_list(gen: SeqGenerator, n: int) -> None:
    seq: Seq = list(islice(gen, n))
    print(seq)
    
def print_matrix(gen: MatGenerator, n: int) -> None:
    mat: Matrix = list(islice(gen, n))
    print(mat)
    
def print_fractions(gen: Generator[frac, None, None], n: int) -> None:
    print(", ".join(str(v) for v in islice(gen, n + 1)))    

In [ ]:
class BackboneMatrix:
    def __init__(self, seq: SeqGenerator, dim: int = 0) -> None:
        """
        Matrix builder with a single source matrix.

        The full matrix stores the triangles:
        - lower triangle values in row slices matrix[i][:i+1]
        - upper triangle values in column slices matrix[:j+1][j]
        - antidiagonal values in matrix[i][d-i] for d = 0..n-1
        Since also the antidiagonal-ups, and antidiagonal-downs can be called,
        this class provides a single source for five representations.
        """
        self.seq_source = seq
        self.matrix: Matrix = []

        if dim > 0: self.grow(dim)

    def grow(self, steps: int = 1) -> None:
        """Expands the matrix dimensions by a given number of steps."""
        for _ in range(steps):
            try:
                r = next(self.seq_source)
            except StopIteration:
                break

            n = len(self.matrix)

            if n == 0:
                self.matrix.append([r])
                continue

            # Build the new lower row from the previous lower row slice and r.
            lower_new = self.matrix[-1][:] + [r]
            for k in range(n, 0, -1):
                lower_new[k - 1] += lower_new[k]

            # Build the new upper column from the previous upper column slice and r.
            prev_upper = [self.matrix[i][n - 1] for i in range(n)]
            upper_new = [0] * n + [r]
            for i in range(n - 1, -1, -1):
                upper_new[i] = upper_new[i + 1] - prev_upper[i]

            # Append the new upper values to existing rows.
            for i in range(n):
                self.matrix[i].append(upper_new[i])

            # Append the new lower row.
            self.matrix.append(lower_new)

    @property
    def backbone(self) -> Seq:
        """Returns the backbone of the matrix, which is the main diagonal."""
        n = len(self.matrix)
        return [self.matrix[i][i] for i in range(n)]
    
    @property
    def binomial_invtrans(self) -> Seq:
        """Returns the inverse binomial transform of the backbone, which is the first row."""
        n = len(self.matrix)
        return [self.matrix[0][i] for i in range(n)]
    
    @property
    def binomial_trans(self) -> Seq:
        """Returns the binomial transform of the backbone, which is the first column."""        
        n = len(self.matrix)
        return [self.matrix[i][0] for i in range(n)]
    
    @property
    def lower_rows(self) -> Matrix:
        """Lower triangle as row slices from the single matrix."""
        return [row[: i + 1] for i, row in enumerate(self.matrix)]

    @property
    def lower_rows_sum(self) -> Seq:
        """Row-wise sums of lower_rows."""
        return [sum(row) for row in self.lower_rows]

    @property
    def upper_rows(self) -> Matrix:
        """Upper triangle as column enumerations from the single matrix."""
        n = len(self.matrix)
        return [[self.matrix[i][j] for i in range(j + 1)] for j in range(n)]

    @property
    def upper_rows_sum(self) -> Seq:
        """Row-wise sums of upper_rows."""
        return [sum(row) for row in self.upper_rows]

    @property
    def diagonals_down(self) -> Matrix:
        """
        Returns antidiagonals in descending columns for d = 0..n-1:
        [matrix[0][d], matrix[1][d-1], ..., matrix[d][0]]
        """
        n = len(self.matrix)
        return [[self.matrix[i][d - i] for i in range(d + 1)] for d in range(n)]

    @property
    def diagonals_down_altsum(self) -> Seq:
        """Row-wise alternating sums of diagonals_down."""
        return [sum((-1)**i * row[i] for i in range(len(row))) for row in self.diagonals_down]

    @property
    def diagonals_up(self) -> Matrix:
        """
        Returns antidiagonals in ascending columns for d = 0..n-1:
        [matrix[d][0], matrix[d-1][1], ..., matrix[0][d]]
        """
        n = len(self.matrix)
        return [[self.matrix[d - i][i] for i in range(d + 1)] for d in range(n)]

    @property
    def diagonals_up_sum(self) -> Seq:
        """Row-wise sums of diagonals_up."""
        return [sum(row) for row in self.diagonals_up]

    def get_matrix(self) -> Matrix:
        """Returns the current state of the matrix."""
        return self.matrix
    
    def get_matrix_dim(self) -> int:
        """Returns the current dimension of the matrix."""
        return len(self.matrix)

    def __iter__(self):
        return self

    def __next__(self) -> Matrix:
        """
        Allows the class instance to be used directly as a generator.
        Yields a deep copy of the matrix at each expansion step.
        """
        before = len(self.matrix)
        self.grow(1)
        if len(self.matrix) == before:
            raise StopIteration
        return [row[:] for row in self.matrix]

#### <a id='toc1_1_1_1_'></a>Short standalone versions

In [ ]:
def binomial_trans(seq: SeqGenerator, dim: int) -> Seq:
    c = []; t = []
    for _ in range(dim):
        try: r = next(seq)
        except StopIteration: break
        c += [r]
        for i in range(len(c) - 1, 0, -1):
            c[i - 1] += c[i]
        t.append(c[0])
    return t

def invbinomial_trans(seq: SeqGenerator, dim: int) -> Seq:
    c = []; t = []
    for _ in range(dim):
        try: r = next(seq)
        except StopIteration: break
        u = [0] * len(c) + [r]
        for i in range(len(c) - 1, -1, -1): 
            u[i] = u[i + 1] - c[i]
        c = u; t.append(u[0])
    return t

### <a id='toc1_1_2_'></a>A display utility for the backbone matrix.

In [ ]:
def Showcase(seq: SeqGenerator, dim: int = 9, verbose: bool = False) -> None:
    """
    Demonstrates the matrix building process.

    Args:
        seq (SeqGenerator): An iterator providing the sequence of integers.
        dim (int): The initial dimension of the matrix to build.
        verbose (bool): If True, prints also antidiagonal triangles.
    """
    builder = BackboneMatrix(seq, dim)
    
    print("Binomial Matrix \n")
    print('|', " | ".join(str(n) for n in range(dim)), '|')
    print('|', "| ".join("- "    for n in range(dim)), '|')
    for row in builder.get_matrix(): 
        print('|', "| ".join(str(v) for v in row), '|')
    
    print("\nMain Diagonal")
    V = (value for value in builder.backbone)
    print('[', ", ".join(str(v) for v in V), ']')
    
    print("\nBinomial Transform")
    V = (value for value in builder.binomial_trans)
    print('[', ", ".join(str(v) for v in V), ']')
    
    print("\nInverse Binomial Transform")
    V = (value for value in builder.binomial_invtrans)
    print('[', ", ".join(str(v) for v in V), ']')
    
    print("\nLower Triangular")
    for row in builder.lower_rows: 
        print('[', ", ".join(str(v) for v in row), ']')

    print("\nLower Triangular Sums:")
    V = (row_sum for row_sum in builder.lower_rows_sum)
    print('[', ", ".join(str(v) for v in V), ']')

    print("\nUpper Triangular")
    for row in builder.upper_rows: 
        print('[', ", ".join(str(v) for v in row), ']')

    print("\nUpper Triangular Sums:")
    V = (row_sum for row_sum in builder.upper_rows_sum)
    print('[', ", ".join(str(v) for v in V), ']')
    
    if verbose:

        print("\nDiagonals Upwards")
        for diags in builder.diagonals_up: 
            print('[', ", ".join(str(v) for v in diags), ']')

        print("\nDiagonals Upwards Sums:")
        V = (row_sum for row_sum in builder.diagonals_up_sum)
        print('[', ", ".join(str(v) for v in V), ']')

        print("\nDiagonals Downwards")
        for diags in builder.diagonals_down: 
            print('[', ", ".join(str(v) for v in diags), ']')
 
        print("\nDiagonals Downwards Alternating Sums:")
        V = (row_sum for row_sum in builder.diagonals_down_altsum)
        print('[', ", ".join(str(v) for v in V), ']')

## <a id='toc1_2_'></a>Riordan

A005043, A126930, A000108, A106640.

In [ ]:
def riordan_generator() -> SeqGenerator:
    b, a, n, r = 1, 0, 1, 0
    yield 1
    while True:
        yield r
        r = n * (2 * a + 3 * b) // (n + 2)
        b, a, n = a, r, n + 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| -1| 2| -3| 6| -10| 20| -35| 70 |
| 1| 0| 1| -1| 3| -4| 10| -15| 35 |
| 2| 1| 1| 0| 2| -1| 6| -5| 20 |
| 5| 3| 2| 1| 2| 1| 5| 1| 15 |
| 14| 9| 6| 4| 3| 3| 6| 6| 16 |
| 42| 28| 19| 13| 9| 6| 9| 12| 22 |
| 132| 90| 62| 43| 30| 21| 15| 21| 34 |
| 429| 297| 207| 145| 102| 72| 51| 36| 55 |
| 1430| 1001| 704| 497| 352| 250| 178| 127| 91 |

In [ ]:
Showcase(riordan_generator())

## <a id='toc1_3_'></a>Catalan

A000108, A005043, A007317, A106640.

In [ ]:
def catalan_generator() -> SeqGenerator:
    c, n = 1, 0
    while True:
        yield c
        c = c * (4 * n + 2) // (n + 2)
        n += 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 1| 1| 3| 6| 15| 36| 91 |
| 2| 1| 1| 2| 4| 9| 21| 51| 127 |
| 5| 3| 2| 3| 6| 13| 30| 72| 178 |
| 15| 10| 7| 5| 9| 19| 43| 102| 250 |
| 51| 36| 26| 19| 14| 28| 62| 145| 352 |
| 188| 137| 101| 75| 56| 42| 90| 207| 497 |
| 731| 543| 406| 305| 230| 174| 132| 297| 704 |
| 2950| 2219| 1676| 1270| 965| 735| 561| 429| 1001 |
| 12235| 9285| 7066| 5390| 4120| 3155| 2420| 1859| 1430 |

In [ ]:
Showcase(catalan_generator())

## <a id='toc1_4_'></a>Motzkin

A001006, A126120, A000108, A058987.

In [ ]:
def motzkin_generator() -> SeqGenerator:
    a, b = 1, 1
    yield a
    yield b
    n = 2
    while True:
        m = ((2 * n + 1) * b + (3 * n - 3) * a) // (n + 2)
        yield m
        a, b, n = b, m, n + 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 1| 0| 2| 0| 5| 0| 14 |
| 2| 1| 1| 1| 2| 2| 5| 5| 14 |
| 5| 3| 2| 2| 3| 4| 7| 10| 19 |
| 14| 9| 6| 4| 5| 7| 11| 17| 29 |
| 42| 28| 19| 13| 9| 12| 18| 28| 46 |
| 132| 90| 62| 43| 30| 21| 30| 46| 74 |
| 429| 297| 207| 145| 102| 72| 51| 76| 120 |
| 1430| 1001| 704| 497| 352| 250| 178| 127| 196 |
| 4862| 3432| 2431| 1727| 1230| 878| 628| 450| 323 |

In [ ]:
Showcase(motzkin_generator())

## <a id='toc1_5_'></a>Fibonacci

A000045, A039834, A001906, A362067, A049601.

In [ ]:
def fibonacci_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 0| 1| -1| 2| -3| 5| -8| 13| -21 |
| 1| 1| 0| 1| -1| 2| -3| 5| -8 |
| 3| 2| 1| 1| 0| 1| -1| 2| -3 |
| 8| 5| 3| 2| 1| 1| 0| 1| -1 |
| 21| 13| 8| 5| 3| 2| 1| 1| 0 |
| 55| 34| 21| 13| 8| 5| 3| 2| 1 |
| 144| 89| 55| 34| 21| 13| 8| 5| 3 |
| 377| 233| 144| 89| 55| 34| 21| 13| 8 |
| 987| 610| 377| 233| 144| 89| 55| 34| 21 |

In [ ]:
Showcase(fibonacci_generator())

## <a id='toc1_6_'></a>Factorial

A000142, A000166, A000522, A002627, A002467.

In [ ]:
def factorial_generator() -> SeqGenerator:
    a, n = 1, 1
    while True:
        yield a
        a, n = a * n, n + 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 1| 2| 9| 44| 265| 1854| 14833 |
| 2| 1| 1| 3| 11| 53| 309| 2119| 16687 |
| 5| 3| 2| 4| 14| 64| 362| 2428| 18806 |
| 16| 11| 8| 6| 18| 78| 426| 2790| 21234 |
| 65| 49| 38| 30| 24| 96| 504| 3216| 24024 |
| 326| 261| 212| 174| 144| 120| 600| 3720| 27240 |
| 1957| 1631| 1370| 1158| 984| 840| 720| 4320| 30960 |
| 13700| 11743| 10112| 8742| 7584| 6600| 5760| 5040| 35280 |
| 109601| 95901| 84158| 74046| 65304| 57720| 51120| 45360| 40320 |

In [ ]:
Showcase(factorial_generator())

## <a id='toc1_7_'></a>Kolakoski

A000002, A054355, A397648.

In [ ]:
def kolakoski_generator() -> SeqGenerator:
    x = y = -1
    while True:
        yield [2, 1][x & 1]
        f = y & ~(y + 1)
        x ^= f
        y = (y + 1) | (f & (x >> 1))

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 1| -1| 0| 2| -4| 3| 8| -43 |
| 3| 2| 0| -1| 2| -2| -1| 11| -35 |
| 7| 4| 2| -1| 1| 0| -3| 10| -24 |
| 14| 7| 3| 1| 0| 1| -3| 7| -14 |
| 26| 12| 5| 2| 1| 1| -2| 4| -7 |
| 48| 22| 10| 5| 3| 2| -1| 2| -3 |
| 91| 43| 21| 11| 6| 3| 1| 1| -1 |
| 178| 87| 44| 23| 12| 6| 3| 2| 0 |
| 357| 179| 92| 48| 25| 13| 7| 4| 2 |

In [ ]:
Showcase(kolakoski_generator())

## <a id='toc1_8_'></a>Bell

A000110, A000296, A005493.

In [ ]:
from itertools import accumulate

def bell_generator() -> SeqGenerator:
    row = [1]
    while True:
        yield row[0]
        row = list(accumulate([row[-1], *row]))

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 1| 1| 4| 11| 41| 162| 715 |
| 2| 1| 1| 2| 5| 15| 52| 203| 877 |
| 5| 3| 2| 3| 7| 20| 67| 255| 1080 |
| 15| 10| 7| 5| 10| 27| 87| 322| 1335 |
| 52| 37| 27| 20| 15| 37| 114| 409| 1657 |
| 203| 151| 114| 87| 67| 52| 151| 523| 2066 |
| 877| 674| 523| 409| 322| 255| 203| 674| 2589 |
| 4140| 3263| 2589| 2066| 1657| 1335| 1080| 877| 3263 |
| 21147| 17007| 13744| 11155| 9089| 7432| 6097| 5017| 4140 |

In [ ]:
Showcase(bell_generator())

## <a id='toc1_9_'></a>Fubini

A000670, A052841, A000629, A089677

In [ ]:
def fubini_generator_a() -> SeqGenerator:
    row = [1]
    total, m = 1, 0

    while True:
        yield total
        m += 1  
        row.append(0)  
        total = 0
        for k in range(m, 0, -1):
            val = k * (row[k - 1] + row[k])
            row[k] = val
            total += val
        row[0] = 0

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 2| 6| 38| 270| 2342| 23646| 272918 |
| 2| 1| 2| 8| 44| 308| 2612| 25988| 296564 |
| 6| 4| 3| 10| 52| 352| 2920| 28600| 322552 |
| 26| 20| 16| 13| 62| 404| 3272| 31520| 351152 |
| 150| 124| 104| 88| 75| 466| 3676| 34792| 382672 |
| 1082| 932| 808| 704| 616| 541| 4142| 38468| 417464 |
| 9366| 8284| 7352| 6544| 5840| 5224| 4683| 42610| 455932 |
| 94586| 85220| 76936| 69584| 63040| 57200| 51976| 47293| 498542 |
| 1091670| 997084| 911864| 834928| 765344| 702304| 645104| 593128| 545835 |

Alternative: this is equivalent, just implemented in C with zip, count and sum. 
/Not/ always faster and requires importing itertools.

In [ ]:
from itertools import count

def fubini_generator() -> SeqGenerator:
    row = [1]
    while True:
        yield sum(row)
        ext = row + [0]   # old row, padded with trailing 0
        row = [0] + [k * (a + b) for k, a, b in zip(count(1), ext, ext[1:])]

In [ ]:
Showcase(fubini_generator())

## <a id='toc1_10_'></a>Central Binomial

A000984, A002426, A026375, A163844, A163774

In [ ]:
def central_binomial_generator() -> SeqGenerator:
    b, n = 1, 0
    while True:
        yield b
        b = b * (4 * n + 2) // (n + 1)
        n += 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 1| 3| 7| 19| 51| 141| 393| 1107 |
| 3| 2| 4| 10| 26| 70| 192| 534| 1500 |
| 11| 8| 6| 14| 36| 96| 262| 726| 2034 |
| 45| 34| 26| 20| 50| 132| 358| 988| 2760 |
| 195| 150| 116| 90| 70| 182| 490| 1346| 3748 |
| 873| 678| 528| 412| 322| 252| 672| 1836| 5094 |
| 3989| 3116| 2438| 1910| 1498| 1176| 924| 2508| 6930 |
| 18483| 14494| 11378| 8940| 7030| 5532| 4356| 3432| 9438 |
| 86515| 68032| 53538| 42160| 33220| 26190| 20658| 16302| 12870 |

In [ ]:
Showcase(central_binomial_generator())

## <a id='toc1_11_'></a>Subfactorial / Rencontres / Derangements

A000166, A000142, A000023, A002467

In [ ]:
def subfactorial_generator() -> SeqGenerator:
    m, a, n = 1, 1, 0
    while True:
        a, m = a * n + m, -m
        yield a
        n += 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| -1| 2| -2| 8| 8| 112| 656| 5504 |
| 1| 0| 1| 0| 6| 16| 120| 768| 6160 |
| 2| 1| 1| 1| 6| 22| 136| 888| 6928 |
| 6| 4| 3| 2| 7| 28| 158| 1024| 7816 |
| 24| 18| 14| 11| 9| 35| 186| 1182| 8840 |
| 120| 96| 78| 64| 53| 44| 221| 1368| 10022 |
| 720| 600| 504| 426| 362| 309| 265| 1589| 11390 |
| 5040| 4320| 3720| 3216| 2790| 2428| 2119| 1854| 12979 |
| 40320| 35280| 30960| 27240| 24024| 21234| 18806| 16687| 14833 |

In [ ]:
Showcase(subfactorial_generator()) 

## <a id='toc1_12_'></a>Involutions (Young tableaux with n cells)

A000085, A005425, A123023, A378100

In [ ]:
def involution_generator() -> SeqGenerator:
    a, b, n = 1, 1, 1
    yield a
    yield b

    while True:
        a, b = b, b + n * a
        n += 1
        yield b

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 1| 0| 3| 0| 15| 0| 105 |
| 2| 1| 1| 1| 3| 3| 15| 15| 105 |
| 5| 3| 2| 2| 4| 6| 18| 30| 120 |
| 14| 9| 6| 4| 6| 10| 24| 48| 150 |
| 43| 29| 20| 14| 10| 16| 34| 72| 198 |
| 142| 99| 70| 50| 36| 26| 50| 106| 270 |
| 499| 357| 258| 188| 138| 102| 76| 156| 376 |
| 1850| 1351| 994| 736| 548| 410| 308| 232| 532 |
| 7193| 5343| 3992| 2998| 2262| 1714| 1304| 996| 764 |

In [ ]:
Showcase(involution_generator()) 

## <a id='toc1_13_'></a>Moebius

A104688, A124839, A008683

In [ ]:
# This is unwise! Moebius(0) is better left undefined. NJAS
#from functools import cache
#@cache
#def mu(n):
#    if n < 2: return n
#    return -sum(mu(d) for d in divisors(n)[:-1])

In [ ]:
from math import isqrt

def moebius_generator() -> SeqGenerator:
    M = [0, 1]
    yield from M

    n = 2
    while True:
        r = isqrt(n)
        s = M[1] 
        
        if n & 1: start, step = 3, 2
        else: start, step = 2, 1

        for d in range(start, r + 1, step):
            q, rem = divmod(n, d)
            if rem == 0:
                s += M[d] + M[q]

        if r * r == n: s -= M[r]

        Mn = -s
        M.append(Mn)
        yield Mn
        n += 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 0| 1| -3| 5| -6| 4| 6| -36| 112 |
| 1| 1| -2| 2| -1| -2| 10| -30| 76 |
| 1| 0| -1| 0| 1| -3| 8| -20| 46 |
| -1| -2| -2| -1| 1| -2| 5| -12| 26 |
| -6| -5| -3| -1| 0| -1| 3| -7| 14 |
| -16| -10| -5| -2| -1| -1| 2| -4| 7 |
| -34| -18| -8| -3| -1| 0| 1| -2| 3 |
| -64| -30| -12| -4| -1| 0| 0| -1| 1 |
| -112| -48| -18| -6| -2| -1| -1| -1| 0 |

In [ ]:
Showcase(moebius_generator()) 

## <a id='toc1_14_'></a>Pell

A000129, A016116, A007052, [A077957, A007070]

In mathematics, the Pell numbers are an infinite sequence of integers, known since ancient times, that comprise the denominators of the closest rational approximations to the square root of 2. This sequence of approximations begins ⁠
1/1⁠, ⁠3/2⁠, ⁠7/5⁠, ⁠17/12⁠, and ⁠41/29⁠, so the sequence of Pell numbers begins with 1, 2, 5, 12, and 29. (Wikipedia)

In [ ]:
def pell_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield b
        a, b = b, a + 2 * b

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 1| 2| 2| 4| 4| 8| 8| 16 |
| 3| 2| 3| 4| 6| 8| 12| 16| 24 |
| 10| 7| 5| 7| 10| 14| 20| 28| 40 |
| 34| 24| 17| 12| 17| 24| 34| 48| 68 |
| 116| 82| 58| 41| 29| 41| 58| 82| 116 |
| 396| 280| 198| 140| 99| 70| 99| 140| 198 |
| 1352| 956| 676| 478| 338| 239| 169| 239| 338 |
| 4616| 3264| 2308| 1632| 1154| 816| 577| 408| 577 |
| 15760| 11144| 7880| 5572| 3940| 2786| 1970| 1393| 985 |

In [ ]:
Showcase(pell_generator()) 

## <a id='toc1_15_'></a>Jacobsthal

A001045, A000244, A020988

In [ ]:
def jacobsthal_generator() -> SeqGenerator:
    a, b = 0, 1
    while True:
        yield a
        a, b = b, b + 2 * a

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 0| 1| -1| 3| -5| 11| -21| 43| -85 |
| 1| 1| 0| 2| -2| 6| -10| 22| -42 |
| 3| 2| 1| 2| 0| 4| -4| 12| -20 |
| 9| 6| 4| 3| 2| 4| 0| 8| -8 |
| 27| 18| 12| 8| 5| 6| 4| 8| 0 |
| 81| 54| 36| 24| 16| 11| 10| 12| 8 |
| 243| 162| 108| 72| 48| 32| 21| 22| 20 |
| 729| 486| 324| 216| 144| 96| 64| 43| 42 |
| 2187| 1458| 972| 648| 432| 288| 192| 128| 85 |

In [ ]:
Showcase(jacobsthal_generator()) 

## <a id='toc1_16_'></a>Polya Trees

A000081.

In [ ]:
# This list version is given for comparison with the generator versions below.

def divisor_table(N: int) -> Matrix:
    """divs[j] = list of divisors of j (increasing), for j = 0..N.
    Built with a sieve in O(N log N) -- no factorization needed."""
    divs = [[] for _ in range(N + 1)]
    for d in range(1, N + 1):
        for j in range(d, N + 1, d):
            divs[j].append(d)
    return divs


def A000081_list(N: int) -> Seq:
    divs = divisor_table(N)
    a = [0] * (N + 1)
    b = [0] * (N + 1)     # b[j] = sum_{d|j} d * a[d]

    if N >= 1:
        a[1] = 1
        b[1] = 1 * a[1]

    for n in range(2, N + 1):
        total = 0
        for j in range(1, n):
            total += b[j] * a[n - j]
        a[n] = total // (n - 1)
        b[n] = sum(d * a[d] for d in divs[n])

    return a

In [ ]:
# Assuming a function Divisors(n) is defined elsewhere, which returns the 
# list of divisors of n. Not as efficient as the sieve version given below, 
# but more direct and readable.

def polyatree_gen() -> SeqGenerator:
    a = [0, 1]
    yield a[0]
    yield a[1]

    n = 2
    while True:
        total = 0
        for j in range(1, n):
            inner = 0
            for d in Divisors(j):
                inner += d * a[d]
            total += inner * a[n - j]

        a_n = total // (n - 1) 
        a.append(a_n)
        yield a_n
        n += 1

In [ ]:
def polyatree_generator() -> SeqGenerator:
    """
    Uses the Divisors[]-style formula:
        a[n] = Sum_j ( Sum_{d|j} d*a[d] ) * a[n-j] / (n-1)
    Divisors are built via an incremental sieve instead of trial-division/factoring.
    """
    a = [0, 1]        # a[0], a[1]
    b = [0, 1]        # b[j] = sum_{d|j} d*a[d];  b[1] = 1*a[1]
    divs = [[], [1]]  # divs[j] = list of divisors of j; divs[1] = [1]

    yield a[0]
    yield a[1]

    n = 1
    while True:
        n += 1
        if n >= len(a):   # grow on demand, amortized doubling
            old_len = len(a)
            new_len = max(n + 1, old_len * 2)
            a.extend([0] * (new_len - old_len))
            b.extend([0] * (new_len - old_len))
            divs.extend([[] for _ in range(new_len - old_len)])

            for d in range(1, new_len):
                start = ((old_len + d - 1) // d) * d  # first multiple of d >= old_len
                for j in range(start, new_len, d):
                    divs[j].append(d)

        # --- a[n] via convolution with b[] ---
        total = 0
        for j in range(1, n):
            total += b[j] * a[n - j]
        a[n] = total // (n - 1)
        b[n] = sum(d * a[d] for d in divs[n])

        yield a[n]

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 0| 1| -1| 2| -2| 4| -5| 13| -25 |
| 1| 1| 0| 1| 0| 2| -1| 8| -12 |
| 3| 2| 1| 1| 1| 2| 1| 7| -4 |
| 8| 5| 3| 2| 2| 3| 3| 8| 3 |
| 22| 14| 9| 6| 4| 5| 6| 11| 11 |
| 64| 42| 28| 19| 13| 9| 11| 17| 22 |
| 195| 131| 89| 61| 42| 29| 20| 28| 39 |
| 615| 420| 289| 200| 139| 97| 68| 48| 67 |
| 1991| 1376| 956| 667| 467| 328| 231| 163| 115 |

In [ ]:
Showcase(polyatree_generator()) 

## <a id='toc1_17_'></a>Euler (alternating permutations, boustrophedon transform)

A000111, A000667, A062162

In [ ]:
def euler_generator() -> SeqGenerator:
    L = [0, 1]      # L[0] = A[-1] = 0, L[1] = A[0] = 1
    offset = 1      # A[k]  <->  L[k + offset]
    k, e, i = 0, 1, 0

    while True:
        Am = 0
        idx = k + e + offset
        if idx == len(L): L.append(0)
        elif idx == -1: L.insert(0, 0); offset += 1
        else: L[idx] = 0
        e = -e

        for _ in range(i + 1):
            pos = k + offset
            Am += L[pos]
            L[pos] = Am
            k += e

        yield Am
        i += 1

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 0| 1| 0| 5| 10| 61| 280 |
| 2| 1| 0| 1| 1| 5| 15| 71| 341 |
| 4| 2| 1| 1| 2| 6| 20| 86| 412 |
| 9| 5| 3| 2| 3| 8| 26| 106| 498 |
| 24| 15| 10| 7| 5| 11| 34| 132| 604 |
| 77| 53| 38| 28| 21| 16| 45| 166| 736 |
| 294| 217| 164| 126| 98| 77| 61| 211| 902 |
| 1309| 1015| 798| 634| 508| 410| 333| 272| 1113 |
| 6664| 5355| 4340| 3542| 2908| 2400| 1990| 1657| 1385 |

In [ ]:
Showcase(euler_generator()) 

## <a id='toc1_18_'></a>Little Schroeder numbers

A001003, A118376, A118376.

In [ ]:
def schroeder_little_generator() -> SeqGenerator:
    b, a, n = 1, 1, 3
    yield b
    yield a

    while True:
        t = a * (6 * n - 9) - (n - 3) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 2| 4| 16| 56| 216| 848| 3424 |
| 2| 1| 2| 6| 20| 72| 272| 1064| 4272 |
| 6| 4| 3| 8| 26| 92| 344| 1336| 5336 |
| 24| 18| 14| 11| 34| 118| 436| 1680| 6672 |
| 112| 88| 70| 56| 45| 152| 554| 2116| 8352 |
| 568| 456| 368| 298| 242| 197| 706| 2670| 10468 |
| 3032| 2464| 2008| 1640| 1342| 1100| 903| 3376| 13138 |
| 16768| 13736| 11272| 9264| 7624| 6282| 5182| 4279| 16514 |
| 95200| 78432| 64696| 53424| 44160| 36536| 30254| 25072| 20793 |

In [ ]:
Showcase(schroeder_little_generator()) 

## <a id='toc1_19_'></a>Big Schroeder numbers

A006318, A174347, A052709

In [ ]:
def schroeder_big_generator() -> SeqGenerator:
    b, a, n = 1, 2, 3
    yield b
    yield a

    while True:
        t = a * (6 * n - 9) - (n - 3) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 1| 3| 9| 31| 113| 431| 1697| 6847 |
| 3| 2| 4| 12| 40| 144| 544| 2128| 8544 |
| 11| 8| 6| 16| 52| 184| 688| 2672| 10672 |
| 47| 36| 28| 22| 68| 236| 872| 3360| 13344 |
| 223| 176| 140| 112| 90| 304| 1108| 4232| 16704 |
| 1135| 912| 736| 596| 484| 394| 1412| 5340| 20936 |
| 6063| 4928| 4016| 3280| 2684| 2200| 1806| 6752| 26276 |
| 33535| 27472| 22544| 18528| 15248| 12564| 10364| 8558| 33028 |
| 190399| 156864| 129392| 106848| 88320| 73072| 60508| 50144| 41586 |

In [ ]:
Showcase(schroeder_big_generator()) 

## <a id='toc1_20_'></a>Central Delannoy

A001850, A080609, A006139.

In [ ]:
def delannoy_generator() -> SeqGenerator:
    b, a, n = 1, 3, 2
    yield b
    yield a

    while True:
        t = a * (6 * n - 3) - (n - 1) * b
        q = t // n
        b, a, n = a, q, n + 1
        yield q

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 2| 8| 32| 136| 592| 2624| 11776| 53344 |
| 4| 3| 10| 40| 168| 728| 3216| 14400| 65120 |
| 20| 16| 13| 50| 208| 896| 3944| 17616| 79520 |
| 112| 92| 76| 63| 258| 1104| 4840| 21560| 97136 |
| 664| 552| 460| 384| 321| 1362| 5944| 26400| 118696 |
| 4064| 3400| 2848| 2388| 2004| 1683| 7306| 32344| 145096 |
| 25376| 21312| 17912| 15064| 12676| 10672| 8989| 39650| 177440 |
| 160640| 135264| 113952| 96040| 80976| 68300| 57628| 48639| 217090 |
| 1027168| 866528| 731264| 617312| 521272| 440296| 371996| 314368| 265729 |

In [ ]:
Showcase(delannoy_generator()) 

## <a id='toc1_21_'></a>Sets of Lists

A000262, A052844, A052845

In [ ]:
def sets_of_lists_generator() -> SeqGenerator:
    b, a, n = 1, 1, 2
    yield b
    yield a

    while True:
        q = (2 * n - 1) * a - (n - 1) * (n - 2) * b
        b, a, n = a, q, n + 1
        yield q

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 2| 6| 36| 240| 1920| 17640| 183120 |
| 2| 1| 2| 8| 42| 276| 2160| 19560| 200760 |
| 6| 4| 3| 10| 50| 318| 2436| 21720| 220320 |
| 26| 20| 16| 13| 60| 368| 2754| 24156| 242040 |
| 148| 122| 102| 86| 73| 428| 3122| 26910| 266196 |
| 1032| 884| 762| 660| 574| 501| 3550| 30032| 293106 |
| 8464| 7432| 6548| 5786| 5126| 4552| 4051| 33582| 323138 |
| 79592| 71128| 63696| 57148| 51362| 46236| 41684| 37633| 356720 |
| 842832| 763240| 692112| 628416| 571268| 519906| 473670| 431986| 394353 |

In [ ]:
Showcase(sets_of_lists_generator()) 

## <a id='toc1_22_'></a>Total Partitions / Ward Set

A000311.

In [ ]:
def total_partitions_generator() -> SeqGenerator:
    yield 0
    yield 1
    row, m = [1], 1

    while True:
        row.append(0)

        for i in range(m, 0, -1):
            row[i] = i * row[i] + (m + i - 1) * row[i - 1]

        row[0] = 0
        m += 1
        yield sum(row)

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 0| 1| -1| 4| 12| 141| 1655| 24116| 411824 |
| 1| 1| 0| 3| 16| 153| 1796| 25771| 435940 |
| 3| 2| 1| 3| 19| 169| 1949| 27567| 461711 |
| 10| 7| 5| 4| 22| 188| 2118| 29516| 489278 |
| 52| 42| 35| 30| 26| 210| 2306| 31634| 518794 |
| 421| 369| 327| 292| 262| 236| 2516| 33940| 550428 |
| 4659| 4238| 3869| 3542| 3250| 2988| 2752| 36456| 584368 |
| 64506| 59847| 55609| 51740| 48198| 44948| 41960| 39208| 620824 |
| 1066048| 1001542| 941695| 886086| 834346| 786148| 741200| 699240| 660032 |

In [ ]:
Showcase(total_partitions_generator()) 

## <a id='toc1_23_'></a>Ward Cycle

A032188

In [ ]:
def wardcycle_generator() -> SeqGenerator:
    yield 1
    yield 1

    n = 1
    row = [0, 1]

    while True:
        n += 1
        row = row + [0]
        for k in range(n, 0, -1):
            row[k] = (n + k - 1) * (row[k - 1] + row[k])
        yield sum(row)

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| - | - | - | - | - | - | - | - | -  |
| 1| 0| 4| 28| 332| 4908| 88556| 1884524| 46219372 |
| 2| 1| 4| 32| 360| 5240| 93464| 1973080| 48103896 |
| 8| 6| 5| 36| 392| 5600| 98704| 2066544| 50076976 |
| 60| 52| 46| 41| 428| 5992| 104304| 2165248| 52143520 |
| 668| 608| 556| 510| 469| 6420| 110296| 2269552| 54308768 |
| 9700| 9032| 8424| 7868| 7358| 6889| 116716| 2379848| 56578320 |
| 172876| 163176| 154144| 145720| 137852| 130494| 123605| 2496564| 58958168 |
| 3648036| 3475160| 3311984| 3157840| 3012120| 2874268| 2743774| 2620169| 61454732 |
| 88918252| 85270216| 81795056| 78483072| 75325232| 72313112| 69438844| 66695070| 64074901 |

In [ ]:
Showcase(wardcycle_generator()) 

## <a id='toc1_24_'></a>Bernoulli numbers

 A164555/A027642, A000367/A002445, A164558, A297703, A014781.

In [ ]:
def bernoulli_seidel() -> Generator[frac, None, None]:
    """Generates Bernoulli numbers (B_0, B_1, B_2, ...) infinitely. B_1 = 1/2."""
    yield frac(1)     # B_0 = 1
    yield frac(1, 2)  # B_1 = 1/2
    
    ZERO = frac(0)  # every odd-indexed Bernoulli number past B_1 vanishes
    row = [1]       # genocchi(0)
    p2 = 8          # 2^(2+1) = 8
    m = 1           # row length
    
    while True:
        # Yield B_n using current Genocchi state
        f = frac(row[-1], p2 - 2)
        yield -f if m % 2 == 0 else f
        
        # Yield B_{n+1} (always 0)
        yield ZERO
        
        # Genocchi triangle rows: print([m-1], row) -> A297703, see also A014781.
        
        # Advance Genocchi state in-place: genocchi(k) -> genocchi(k + 1)
        row.append(0)
        m += 1
        
        # Right-to-left accumulation
        for k in range(m - 2, -1, -1): 
            row[k] += row[k + 1]
            
        # Left-to-right accumulation
        for k in range(1, m): 
            row[k] += row[k - 1]

        p2 <<= 2

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
| - | - | - | - | - | - | - | - | - | -  |
| 1| -1/2| 1/6| 0| -1/30| 0| 1/42| 0| -1/30| 0 |
| 3/2| 1/2| -1/3| 1/6| -1/30| -1/30| 1/42| 1/42| -1/30| -1/30 |
| 13/6| 2/3| 1/6| -1/6| 2/15| -1/15| -1/105| 1/21| -1/105| -1/15 |
| 3| 5/6| 1/6| 0| -1/30| 1/15| -8/105| 4/105| 4/105| -8/105 |
| 119/30| 29/30| 2/15| -1/30| -1/30| 1/30| -1/105| -4/105| 8/105| -4/105 |
| 5| 31/30| 1/15| -1/15| -1/30| 0| 1/42| -1/21| 4/105| 4/105 |
| 253/42| 43/42| -1/105| -8/105| -1/105| 1/42| 1/42| -1/42| -1/105| 8/105 |
| 7| 41/42| -1/21| -4/105| 4/105| 1/21| 1/42| 0| -1/30| 1/15 |
| 239/30| 29/30| -1/105| 4/105| 8/105| 4/105| -1/105| -1/30| -1/30| 1/30 |
| 9| 31/30| 1/15| 8/105| 4/105| -4/105| -8/105| -1/15| -1/30| 0 |


In [ ]:
from timeit import default_timer as timer

def test(M: int) -> None:

    # Print B_0, B_2, B_4, ..., B_14 (every 2nd term up to index 14)
    # Total terms up to index 14 is 15 items: [B_0, B_1, B_2, ..., B_14]
    print_fractions(bernoulli_seidel(), 14)

    # Benchmark computing up to M = 1000 terms (B_0 through B_1000)
    start = timer()
    
    # Consumes the generator up to term M+1
    _ = list(islice(bernoulli_seidel(), M + 1))
    
    end = timer()
    print(f"Time taken for M = {M}: {end - start:.6f} seconds")

test(1000)  # For M = 1000 this takes on my machine between 0.2 and 0.3 seconds.

In [ ]:
Showcase(bernoulli_seidel(), 10)  # ignore the type warning

Numerators only

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
| - | - | - | - | - | - | - | - | - | -  |
| 1| -1| 1| 0| -1| 0| 1| 0| -1| 0 |
| 3| 1| -1| 1| -1| -1| 1| 1| -1| -1 |
| 13| 2| 1| -1| 2| -1| -1| 1| -1| -1 |
| 3| 5| 1| 0| -1| 1| -8| 4| 4| -8 |
| 119| 29| 2| -1| -1| 1| -1| -4| 8| -4 |
| 5| 31| 1| -1| -1| 0| 1| -1| 4| 4 |
| 253| 43| -1| -8| -1| 1| 1| -1| -1| 8 |
| 7| 41| -1| -4| 4| 1| 1| 0| -1| 1 |
| 239| 29| -1| 4| 8| 4| -1| -1| -1| 1 |
| 9| 31| 1| 8| 4| -4| -8| -1| -1| 0 |



## Appendix: some matrix generators

In [ ]:
def A060187_mat() -> MatGenerator:
    row = [1]  
    yield list(row)
    n = 1
    while True:
        row.append(1)
        for k in range(n-1, 0, -1):
            row[k] = (2 * (n - k) + 1) * row[k - 1] + (2 * k + 1) * row[k]
        yield list(row)
        n += 1
        
print_matrix(A060187_mat(), n=6)

In [ ]:
def A028338_mat() -> MatGenerator:
    yield list([1])
    n = 1; row = [0, 1]
    while True:
        row.append(1)
        for m in range(n, 0, -1):
            row[m] = (2 * n - 1) * row[m] + row[m - 1]
        yield row[1:]
        n += 1

print_matrix(A028338_mat(), n=6)

# BENCHMARK

In [ ]:
import time

class StopWatch:
    def __init__(
        self,
        comment: str = "elapsed time"
    ) -> None:
        self.start_time = None
        self.text = comment

    def start(self) -> None:
        """Start a new StopWatch"""
        if self.start_time is not None:
            raise RuntimeError("Watch is running. First stop it.")
        self.start_time = time.perf_counter()

    def stop(self) -> float:
        """Stop the StopWatch, and report the elapsed time."""
        if self.start_time is None:
            raise RuntimeError("Watch is not running.")

        elapsed_time = time.perf_counter() - self.start_time
        self.start_time = None

        print(self.text.rjust(17), "{:0.4f}".format(elapsed_time), "sec")

        return elapsed_time


def Benchmark(gen: SeqGenerator, 
              offset:int = 8, 
              size:int = 4
    ) -> None:
    """Benchmark for sequence generators.

    Args:
        gen, sequence generator
        offset > 0, the power of two where the test starts. Defaults to 4.
        size, the length of test run. Defaults to 4.

    Returns:
        List of elapsed time. 
        Stops if the computing time exceeds 1 second.

    Example:
        Benchmark(lambda n, k: n**k)
    """
    print("\n", gen.__qualname__)
    B: list[float] = []
    for s in [2 << n for n in range(offset - 1, offset + size)]:
        t = StopWatch(str(s))
        t.start()
        list(islice(gen, s))
        e = t.stop()
        B.append(e)
        if e > 1.0: break
    return None

In [245]:
Benchmark(riordan_generator())
Benchmark(catalan_generator())
Benchmark(motzkin_generator())
Benchmark(fibonacci_generator())
Benchmark(factorial_generator())
Benchmark(kolakoski_generator())
Benchmark(bell_generator())
Benchmark(fubini_generator_a())
Benchmark(fubini_generator())
Benchmark(central_binomial_generator())
Benchmark(subfactorial_generator())
Benchmark(involution_generator())
Benchmark(moebius_generator())
Benchmark(pell_generator())
Benchmark(jacobsthal_generator())
Benchmark(polyatree_generator())
Benchmark(euler_generator())
Benchmark(schroeder_little_generator())
Benchmark(schroeder_big_generator())
Benchmark(delannoy_generator())
Benchmark(sets_of_lists_generator())
Benchmark(total_partitions_generator())
Benchmark(wardcycle_generator())


 riordan_generator
              256 0.0005 sec
              512 0.0013 sec
             1024 0.0030 sec
             2048 0.0155 sec
             4096 0.0491 sec

 catalan_generator
              256 0.0002 sec
              512 0.0006 sec
             1024 0.0025 sec
             2048 0.0791 sec
             4096 0.0501 sec

 motzkin_generator
              256 0.0002 sec
              512 0.0007 sec
             1024 0.0036 sec
             2048 0.0163 sec
             4096 0.0527 sec

 fibonacci_generator
              256 0.0001 sec
              512 0.0001 sec
             1024 0.0003 sec
             2048 0.0012 sec
             4096 0.0035 sec

 factorial_generator
              256 0.0001 sec
              512 0.0005 sec
             1024 0.0018 sec
             2048 0.0083 sec
             4096 0.0408 sec

 kolakoski_generator
              256 0.0003 sec
              512 0.0006 sec
             1024 0.0011 sec
             2048 0.0022 sec
             4096 0.0045 sec

 be